# 05 - Results

Reads your hand-check verdicts, builds the final ensembles and leaderboards, and prints the go/no-go
decision. **Cost: $0** (no Gemini calls). Takes a few minutes. Safe to rerun after editing the sheet.

**Step 1 - setup (free).** Downloads the project code from GitHub, installs it, connects Google Drive
(all results are saved in `MyDrive/cfpb-nlp-classifier/phase1`), reads your `GEMINI_API_KEY` secret (and the optional `TYPESAFE_API_KEY` Jev secret) and
logs in to Google Sheets. Colab will ask you to allow Drive and Sheets access - click Allow.
Takes 2-4 minutes the first time. If it says the key is missing: click the key icon on the left of
Colab, add a secret named `GEMINI_API_KEY`, and switch "Notebook access" on.

In [ ]:
!git clone -q https://github.com/Dentoneer/cfpb-nlp-classifier /content/cfpb-nlp-classifier || git -C /content/cfpb-nlp-classifier pull -q
%pip install -q -e "/content/cfpb-nlp-classifier[ml,colab]"
import sys; sys.path.insert(0, "/content/cfpb-nlp-classifier")
from nlp.colab import setup
cfg, gc, api_key, jev_key = setup()

**Load saved results (free).**

In [ ]:
import pandas as pd
from nlp import colab, store, pipeline as P
from nlp.ensemble import row_status
complaints = store.read(cfg, P.STAGE_COMPLAINTS)
splits = store.read(cfg, P.STAGE_SPLITS)
gem = store.read(cfg, P.STAGE_PRED_GEMINI)
selection = store.read(cfg, P.STAGE_HAND_CHECK)
vres = store.read(cfg, P.STAGE_VALIDATORS)
train_df, test_df = P.train_test_frames(complaints, splits)
ledger, *_ = colab.open_tools(cfg)

**Read your verdicts from the sheet.** **Look at:** the counts table. If it says some verdict cells are
still blank, finish the sheet and rerun this cell. (Set `ALLOW_PARTIAL = True` only to score a
partly-filled sheet.)

In [ ]:
ALLOW_PARTIAL = False
url = colab.read_url(cfg, "review_sheet")
human = colab.load_human_review(cfg, gc, url, allow_partial=ALLOW_PARTIAL)

**Ensembles and row status (free).** Combines all validators into one score per complaint (a simple
average and one learned from your verdicts), and labels each test row `verified`, `likely_correct` or
`needs_review`.

In [ ]:
write = colab.always_write(cfg)
ens_simple, ens_learned = P.run_ensembles(vres, human)
write(P.STAGE_ENSEMBLE_SIMPLE, lambda: ens_simple)
write(P.STAGE_ENSEMBLE_LEARNED, lambda: ens_learned)
status = write(P.STAGE_ROW_STATUS, lambda: row_status(gem, complaints, ens_learned, cfg=cfg))
print(status["status"].value_counts())

**Leaderboards, adjusted accuracy and decision (free).** Scores every classifier against the CFPB labels,
computes Gemini's accuracy from your verdicts (reweighted to the whole test set), ranks the validators and
applies the go/no-go rule. All tables are saved to Drive as Parquet.

In [ ]:
preds = [gem] + [store.read(cfg, P.stage_pred(n)) for n in ["tfidf", "embed_lr", "knn", "modernbert"]]
scored = P.run_scoring(cfg, pd.concat(preds, ignore_index=True), complaints,
    test_df["complaint_id"].tolist(), gem, human, selection,
    P.all_validator_results(vres, ens_simple, ens_learned), n_boot=1000, stage=write)
colab.list_outputs(cfg)

**Results summary.** Below: the decision with reasons, adjusted accuracy, both leaderboards and the total
Gemini spend. A copy is saved as `results.md` on Drive.

In [ ]:
colab.show_results(cfg, scored, ledger)
print(colab.ledger_report(ledger))